# 04 — Ensembles + GridSearchCV
Voting (Hard/Soft), Stacking, GridSearchCV on top 3

**Optimized:** tqdm, reduced grids, subsampled ensembles, 3-fold CV

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, pickle, os, time
from tqdm.notebook import tqdm
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, StackingClassifier
SEED=42; PROC='data/processed'; MOD='models'
os.makedirs('charts/model_comparison',exist_ok=True)
print('Ready')

In [ ]:
X=pd.read_csv(f'{PROC}/X_processed.csv')
y=pd.read_csv(f'{PROC}/y_target.csv').iloc[:,0]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
cv=StratifiedKFold(3,shuffle=True,random_state=SEED)
cv_results=pickle.load(open(f'{MOD}/cv_results.pkl','rb'))
top5=sorted(cv_results,key=lambda k: cv_results[k]['f1'],reverse=True)[:5]
print(f'Top 5: {top5}')

# Subsample for ensembles (50k) to speed up
ENS_SAMPLE=50000
idx_ens=np.random.RandomState(SEED).choice(len(X_train),min(ENS_SAMPLE,len(X_train)),replace=False)
X_train_ens=X_train.iloc[idx_ens]; y_train_ens=y_train.iloc[idx_ens]
print(f'Ensemble subsample: {len(X_train_ens):,}')

## 1. Fresh Model Factory

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

def fresh(name):
    m={
        'logistic_regression': LogisticRegression(C=1,max_iter=1000,solver='lbfgs',class_weight='balanced',random_state=SEED,n_jobs=-1),
        'knn': KNeighborsClassifier(n_neighbors=7,weights='distance',n_jobs=-1),
        'gaussian_nb': GaussianNB(),
        'sgd_classifier': SGDClassifier(loss='log_loss',alpha=0.001,class_weight='balanced',random_state=SEED,n_jobs=-1),
        'decision_tree': DecisionTreeClassifier(max_depth=10,class_weight='balanced',random_state=SEED),
        'random_forest': RandomForestClassifier(n_estimators=100,max_depth=12,class_weight='balanced',random_state=SEED,n_jobs=-1),
        'extra_trees': ExtraTreesClassifier(n_estimators=100,max_depth=12,class_weight='balanced',random_state=SEED,n_jobs=-1),
        'adaboost': AdaBoostClassifier(n_estimators=80,learning_rate=0.1,random_state=SEED),
        'gradient_boosting': GradientBoostingClassifier(n_estimators=80,learning_rate=0.1,max_depth=5,random_state=SEED),
        'xgboost': XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=6,eval_metric='mlogloss',use_label_encoder=False,random_state=SEED,n_jobs=-1,tree_method='hist'),
        'lightgbm': LGBMClassifier(n_estimators=100,learning_rate=0.1,num_leaves=31,class_weight='balanced',random_state=SEED,verbose=-1,n_jobs=-1),
        'catboost': CatBoostClassifier(iterations=100,learning_rate=0.1,depth=6,random_seed=SEED,verbose=0),
        'mlp': MLPClassifier(hidden_layer_sizes=(64,32),activation='relu',solver='adam',max_iter=300,early_stopping=True,random_state=SEED),
    }
    return m[name]

## 2. Voting Ensembles (tqdm)

In [ ]:
ens_trained={}
for vote_type in tqdm(['soft','hard'],desc='Voting Ensembles'):
    t0=time.time()
    est=[(n,fresh(n)) for n in top5]
    vc=VotingClassifier(estimators=est,voting=vote_type,n_jobs=-1)
    sc=cross_val_score(vc,X_train_ens,y_train_ens,cv=cv,scoring='f1_macro',n_jobs=-1)
    vc.fit(X_train_ens,y_train_ens)
    ens_trained[f'voting_{vote_type}']=vc
    tqdm.write(f'  ✓ voting_{vote_type}: F1={sc.mean():.4f} ({time.time()-t0:.1f}s)')

## 3. Stacking

In [ ]:
t0=time.time()
stack_est=[(n,fresh(n)) for n in top5[:3]]
stack=StackingClassifier(estimators=stack_est,final_estimator=LogisticRegression(max_iter=1000,random_state=SEED),cv=3,n_jobs=-1)
sc=cross_val_score(stack,X_train_ens,y_train_ens,cv=cv,scoring='f1_macro',n_jobs=-1)
stack.fit(X_train_ens,y_train_ens)
ens_trained['stacking']=stack
print(f'  ✓ stacking: F1={sc.mean():.4f} ({time.time()-t0:.1f}s)')

## 4. GridSearchCV (Reduced Grids, tqdm)

In [ ]:
# Use 40k subsample for GridSearch
GS_SAMPLE=40000
idx_gs=np.random.RandomState(SEED).choice(len(X_train),min(GS_SAMPLE,len(X_train)),replace=False)
X_train_gs=X_train.iloc[idx_gs]; y_train_gs=y_train.iloc[idx_gs]
print(f'GridSearch subsample: {len(X_train_gs):,}')

grids={
    'random_forest':{'n_estimators':[100,200],'max_depth':[10,15]},
    'xgboost':{'n_estimators':[100,200],'learning_rate':[0.05,0.1],'max_depth':[4,6]},
    'lightgbm':{'n_estimators':[100,200],'learning_rate':[0.05,0.1],'num_leaves':[15,31]},
}
top3=top5[:3]
for name in tqdm(top3,desc='GridSearchCV'):
    if name in grids:
        t0=time.time()
        gs=GridSearchCV(fresh(name),grids[name],cv=3,scoring='f1_macro',n_jobs=-1)
        gs.fit(X_train_gs,y_train_gs)
        ens_trained[f'{name}_tuned']=gs.best_estimator_
        tqdm.write(f'  ✓ {name}_tuned: F1={gs.best_score_:.4f} params={gs.best_params_} ({time.time()-t0:.1f}s)')
    else:
        tqdm.write(f'  ⊘ {name}: no grid defined')

## 5. Save

In [ ]:
for n,m in tqdm(ens_trained.items(),desc='Saving Ensembles'):
    pickle.dump(m,open(f'{MOD}/model_zoo/{n}.pkl','wb'))
print(f'Saved {len(ens_trained)} ensemble/tuned models')